# Example 8.9.2 / 2: HiGHS MIP Search Options

Companion notebook for **8.9.2 LP and MIP Algorithm Options** in the AMPL lab Documentation, section "HiGHS MIP Search Options". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## HiGHS MIP Search Options

This example runs the knapsack model of How MIP Solvers Work (AMPL lab documentation, section 8.9.1) with HiGHS options
that change a visible part of the search, one at a time, and prints the status, incumbent and
the work from `solve_message`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
# packed weight within capacity
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;
# packed volume within capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
ampl.option["solver_msg"] = 0
runs = ["", "pre:solve=off", "mip:heureff=0", "mip:pscostreliability=0",
        "mip:poolsoftlimit=1", "lim:leavenodes=1", "lim:improvingsols=1",
        "lim:objectivetarget=150", "mip:gapabs=20"]
for opts in runs:
    ampl.solve(solver="highs", highs_options=opts)
    work = ", ".join(ampl.get_value("solve_message").splitlines()[1:])
    incumbent = ampl.obj["Total_Value"].value()
    print(f"{opts or 'defaults'}: {ampl.solve_result}, {incumbent:g}; {work}")

defaults: solved, 153; 68 simplex iterations, 6 branching nodes
HiGHS 1.14.0:   pre:solve = off
pre:solve=off: solved, 153; 68 simplex iterations, 6 branching nodes
HiGHS 1.14.0:   mip:heureff = 0
mip:heureff=0: solved, 153; 68 simplex iterations, 6 branching nodes
HiGHS 1.14.0:   mip:pscostreliability = 0
mip:pscostreliability=0: solved, 153; 45 simplex iterations, 16 branching nodes
HiGHS 1.14.0:   mip:poolsoftlimit = 1
mip:poolsoftlimit=1: solved, 153; 64 simplex iterations, 6 branching nodes
HiGHS 1.14.0:   lim:leavenodes = 1
  Warning code 1 while solving with HiGHS
lim:leavenodes=1: limit, 153; 51 simplex iterations, 3 branching nodes, absmipgap=11, relmipgap=0.0718954
HiGHS 1.14.0:   lim:improvingsols = 1
  Warning code 1 while solving with HiGHS
lim:improvingsols=1: limit, 138; 3 simplex iterations, 0 branching nodes, absmipgap=40, relmipgap=0.289855
HiGHS 1.14.0:   lim:objectivetarget = 150
lim:objectivetarget=150: limit, 153; 17 simplex iterations, 1 branching nodes, absmipga

### Pure AMPL: data.dat and model.run

The same loop in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
# HiGHS option settings to compare; "" means defaults
set RUNS ordered = {"", "pre:solve=off", "mip:heureff=0", "mip:pscostreliability=0",
    "mip:poolsoftlimit=1", "lim:leavenodes=1", "lim:improvingsols=1",
    "lim:objectivetarget=150", "mip:gapabs=20"};
for {opts in RUNS} {
    option highs_options (opts);
    solve;
    printf "%s: %s, %g\n", if opts = "" then "defaults" else opts, solve_result, Total_Value;
}

defaults: solved, 153
HiGHS 1.14.0:   pre:solve = off
pre:solve=off: solved, 153
HiGHS 1.14.0:   mip:heureff = 0
mip:heureff=0: solved, 153
HiGHS 1.14.0:   mip:pscostreliability = 0
mip:pscostreliability=0: solved, 153
HiGHS 1.14.0:   mip:poolsoftlimit = 1
mip:poolsoftlimit=1: solved, 153
HiGHS 1.14.0:   lim:leavenodes = 1
  Warning code 1 while solving with HiGHS
lim:leavenodes=1: limit, 153
HiGHS 1.14.0:   lim:improvingsols = 1
  Warning code 1 while solving with HiGHS
lim:improvingsols=1: limit, 138
HiGHS 1.14.0:   lim:objectivetarget = 150
lim:objectivetarget=150: limit, 153
HiGHS 1.14.0:   mip:gapabs = 20
mip:gapabs=20: solved, 153


### Another way in amplpy

The table lists the MIP search options in the installed HiGHS 1.14.0 listing, grouped by the
part of the search they steer. "Result on this model" comes from the loop above or from the
same model run with that option alone; "no change" means 6 nodes and 68 simplex iterations, as
with the defaults. Rows marked "previous page" are run on How MIP Solvers Work (AMPL lab documentation, section 8.9.1).

| Part | Option (default) | Documented effect | Result on this model |
|---|---|---|---|
| Presolve | `pre:solve` (`choose`; also `off`, `on`) | Whether to use presolve | `off`: no change; presolve removed nothing here |
| Heuristics | `mip:heureff` (`0.05`) | Fraction of time to spend in MIP heuristics | `0`: no change |
| Branching | `mip:pscostreliability` (`8`) | Minimal number of observations before pseudo costs are considered reliable | `0`: 16 nodes and 45 iterations instead of 6 and 68 |
| Cuts | `mip:poolsoftlimit` (`10000`) | Soft limit on the number of rows in the cut pool | `1`: 64 iterations instead of 68 |
| Cuts | `mip:lpagelimit` (`10`) | Maximal age of dynamic LP rows before they are removed from the relaxation | `1`: no change |
| Presolve | `mip:lifting` (`-1`) | Whether lifting for probing should be used | `0`: no change |
| Symmetry | `mip:detsimmetry` (`1`) | Whether symmetry should be detected | `0`: no change |
| Parallelism | `mip:mincliquetable` (`100000`) | Minimal clique-table entries before conflict-graph queries run in parallel | `1`: no change |
| Integrality | `mip:intfeastol` (`1e-06`) | Feasibility tolerance for integer variables | `1e-5`: no change |
| Stopping | `lim:nodes` (no limit) | Maximum MIP nodes to explore | `1`: `limit`, bound 168 (previous page) |
| Stopping | `lim:leavenodes` (no limit) | Maximum number of leaf nodes | `1`: `limit` after 3 nodes, gap 11 |
| Stopping | `lim:stallnodes` (no limit) | Maximum nodes where the estimate is above the cutoff bound | `1`: no change |
| Stopping | `lim:improvingsols` (no limit) | Maximum number of improving solutions found | `1`: `limit` at incumbent 138, gap 40 |
| Stopping | `lim:objectivetarget` (no limit) | Objective target for termination of the MIP solver | `150`: `limit` at 153 after 1 node |
| Stopping | `mip:gap` (`1e-04`) | Relative gap tolerance for declaring optimality | `0.1`: `solved` with gap 15 (previous page) |
| Stopping | `mip:gapabs` (`1e-06`) | Absolute gap tolerance for declaring optimality | `20`: `solved` with gap 15 after 1 node |
| Reporting | `mip:bestbound` (`0`) | Return `.bestbound` | `1`: bound 168 or 153 (previous page) |
| Reporting | `mip:return_gap` (`0`) | Return `.relmipgap` (`1`), `.absmipgap` (`2`); `4` suppresses gaps in the message | `3`: both suffixes (previous page) |
| Reporting | `tech:outlev` (`0`) | Write HiGHS log lines | `1`: the log (previous page) |
| Reporting | `tech:miploglev` (`1`) | MIP solver report level (`0`, `1`, `2`) | `2`: no change in the result |
| Relaxation | `alg:relax` (`0`) | Whether to relax integrality of variables | `1`: relaxation value 178.0945 (previous page) |

`mip:round` and `mip:round_reptol` change how integer values are reported, not the search, and
Numerical Accuracy (AMPL lab documentation, section 8.7) covers integrality tolerances. Time limits and plateau stopping are
in Stopping Criteria and Reproducible Runs (AMPL lab documentation, section 8.2.2).

**Pitfall: an option newer than the installed driver.** The live HiGHS options page, checked on
2026-10-28, also documents `mip:lpsolver` (the LP
solver used during MIP solves: `choose`, `simplex`, `ipm`, `hipo`, `ipx`) and `mip:ipmsolver`
(`choose`, `ipm`, `hipo`). The HiGHS 1.14.0 driver used on this page does not know them. With
the same `model.mod` and `data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [11]:
# run.py
ampl.solve(solver="highs", highs_options="mip:lpsolver=simplex")
print(ampl.solve_result)

HiGHS 1.14.0: Unknown option or invalid key "mip:lpsolver"
failure


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod` and `data.dat`:

In [12]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option highs_options 'mip:lpsolver=simplex';
solve;
display solve_result;

HiGHS 1.14.0: Unknown option or invalid key "mip:lpsolver"
solve_result = failure

